In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re

from sklearn.model_selection import (
    train_test_split, cross_val_score, StratifiedKFold
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import RFE
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score

import lightgbm as lgb

print("OK")

OK


In [2]:
url = "https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv"
df = pd.read_csv(url)

df["Title"] = df["Name"].str.extract(r" ([A-Za-z]+)\.")
common_titles = ["Mr", "Miss", "Mrs", "Master"]
df["Title"] = df["Title"].where(df["Title"].isin(common_titles), "Rare")

# Пропуски
df["Age"] = df["Age"].fillna(df["Age"].median())
df["Embarked"] = df["Embarked"].fillna(df["Embarked"].mode()[0])

# Новые фичи
df["FamilySize"] = df["SibSp"] + df["Parch"] + 1
df["IsAlone"] = (df["FamilySize"] == 1).astype(int)

print("Форма:", df.shape)
print("Пропусков:", df.isna().sum().sum())


Форма: (891, 15)
Пропусков: 687


In [3]:
print("Пропусков:", df.isna().sum().sum())
df.head()

Пропусков: 687


,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked,Title,FamilySize,IsAlone
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S,Mr,2,0
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C,Mrs,2,0
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S,Miss,1,1
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S,Mrs,2,0
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S,Mr,1,1


In [4]:
df_base = df[['Survived', 'Pclass', 'Age', 'Sex', 'Fare',
              'Embarked', 'Title', 'FamilySize', 'IsAlone']].copy()
df_base = pd.get_dummies(df_base, columns=['Sex', 'Embarked', 'Title'], drop_first=False)

X_base = df_base.drop(columns=['Survived'])
y = df_base['Survived']

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

lgb_base = lgb.LGBMClassifier(n_estimators=200, max_depth=3,
                               learning_rate=0.1, random_state=42, verbose=-1)
scores_base = cross_val_score(lgb_base, X_base, y, cv=cv, scoring="accuracy")

print(f"BASELINE: mean={scores_base.mean():.4f}, std={scores_base.std():.4f}")
print(f"Признаков: {X_base.shape[1]}")

BASELINE: mean=0.8440, std=0.0165
Признаков: 15


In [5]:
df_new = df.copy()

df_new['Age_bin'] = pd.cut(
    df_new['Age'],
    bins=[0, 12, 18, 60, 100],
    labels=['ребёнок', 'подросток', 'взрослый', 'пожилой']
)

df_new["Fare_per_person"] = df_new["Fare"]/df_new["FamilySize"]

df_new["IsChild"] = (df_new["Age"] < 12).astype(int)

df_new["Pclass_x_FamilySize"] = df_new["Pclass"] * df_new["FamilySize"]

fare_mean_by_class = df_new.groupby("Pclass")["Fare"].transform("mean")
df_new["Fare_relative"] = df_new["Fare"] / fare_mean_by_class

df_new.head()
print(df_new[["Age_bin", "Fare_per_person", "IsChild", "Pclass_x_FamilySize", "Fare_relative"]].head(10))

     Age_bin  Fare_per_person  IsChild  Pclass_x_FamilySize  Fare_relative
0   взрослый          3.62500        0                    6       0.530143
1   взрослый         35.64165        0                    2       0.847051
2   взрослый          7.92500        0                    3       0.579501
3   взрослый         26.55000        0                    2       0.630981
4   взрослый          8.05000        0                    3       0.588642
5   взрослый          8.45830        0                    3       0.618498
6   взрослый         51.86250        0                    1       0.616276
7    ребёнок          4.21500        1                   15       1.541071
8   взрослый          3.71110        0                    9       0.814103
9  подросток         15.03540        0                    4       1.455354


In [6]:
df_new = df_new.drop(columns=['Name', 'SibSp', 'Parch', 'Ticket', 'Cabin', 'PassengerId'])
df_new = df_new[["Survived", "Pclass", "Sex", "Age", "Fare",
                 "Embarked", "Title", "FamilySize", "IsAlone",
                 "Age_bin", "Fare_per_person", "IsChild",
                 "Pclass_x_FamilySize", "Fare_relative"]]
df_new=pd.get_dummies(df_new, columns=["Sex", "Embarked", "Title", "Age_bin"], drop_first=False)

X_new = df_new.drop(columns=["Survived"])
y_new = df_new["Survived"]

print("Признаков теперь:", X_new.shape[1])
print("Новые колнки:", [c for c in X_new.columns if "Age_bin" in c or "Fare_" in c or c == "IsChild" or c== "Pclass_x"])

Признаков теперь: 23
Новые колнки: ['Fare_per_person', 'IsChild', 'Fare_relative', 'Age_bin_ребёнок', 'Age_bin_подросток', 'Age_bin_взрослый', 'Age_bin_пожилой']


In [8]:
df_new.head()

,Survived,Pclass,Age,Fare,FamilySize,IsAlone,Fare_per_person,IsChild,Pclass_x_FamilySize,Fare_relative,...,Embarked_S,Title_Master,Title_Miss,Title_Mr,Title_Mrs,Title_Rare,Age_bin_ребёнок,Age_bin_подросток,Age_bin_взрослый,Age_bin_пожилой
0,0,3,22.0,7.2500,2,0,3.62500,0,6,0.530143,...,True,False,False,True,False,False,False,False,True,False
1,1,1,38.0,71.2833,2,0,35.64165,0,2,0.847051,...,False,False,False,False,True,False,False,False,True,False
2,1,3,26.0,7.9250,1,1,7.92500,0,3,0.579501,...,True,False,True,False,False,False,False,False,True,False
3,1,1,35.0,53.1000,2,0,26.55000,0,2,0.630981,...,True,False,False,False,True,False,False,False,True,False
4,0,3,35.0,8.0500,1,1,8.05000,0,3,0.588642,...,True,False,False,True,False,False,False,False,True,False


In [11]:
lgb_new = lgb.LGBMClassifier(n_estimators=200, max_depth=3, learning_rate=0.1, random_state=42, verbose=-1)
scores_new=cross_val_score(lgb_new, X_new, y_new, cv=cv, scoring="accuracy")

print(f"BASELINE:  mean={scores_base.mean():.4f}, std={scores_base.std():.4f}")
print(f"С НОВЫМИ:  mean={scores_new.mean():.4f}, std={scores_new.std():.4f}")
print(f"Прирост:   {scores_new.mean() - scores_base.mean():+.4f}")

BASELINE:  mean=0.8440, std=0.0165
С НОВЫМИ:  mean=0.8406, std=0.0129
Прирост:   -0.0034


In [15]:
pipe_lr_new = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(max_iter=1000)),
])

pipe_lr_base = Pipeline([
    ("scaler", StandardScaler()),
    ('model', LogisticRegression(max_iter=1000)),
])

scores_lr_base = cross_val_score(pipe_lr_base, X_base, y, cv=cv, scoring="accuracy")
scores_lr_new=cross_val_score(pipe_lr_new, X_new, y_new, cv=cv, scoring="accuracy")

print(f"LR baseline:  mean={scores_lr_base.mean():.4f}, std={scores_lr_base.std():.4f}")
print(f"LR c новыми:  mean={scores_lr_new.mean():.4f},  std={scores_lr_new.std():.4f}")
print(f"Прирост:      {scores_lr_new.mean() - scores_lr_base.mean():+.4f}")

LR baseline:  mean=0.8272, std=0.0091
LR c новыми:  mean=0.8350,  std=0.0144
Прирост:      +0.0079


In [16]:
lgb_new.fit(X_new, y_new)

importances = pd.Series(
    lgb_new.feature_importances_,
    index=X_new.columns
).sort_values(ascending=False)

print(importances.head(15))

Fare_per_person        234
Age                    229
Fare                   170
Fare_relative          146
Pclass_x_FamilySize     42
Title_Master            38
Sex_female              38
FamilySize              28
Title_Mrs               23
Pclass                  23
Embarked_C              21
Title_Mr                20
Embarked_S              19
Title_Rare              11
Age_bin_взрослый         9
dtype: int32


In [17]:
lgb_base.fit(X_base, y)
importances_base = pd.Series(lgb_base.feature_importances_, index=X_base.columns).sort_values(ascending=False)
print(importances_base.head(10))

Fare            396
Age             303
FamilySize       87
Pclass           52
Title_Mr         41
Sex_female       31
Title_Master     29
Title_Mrs        25
Title_Rare       23
Embarked_S       17
dtype: int32


In [18]:
from sklearn.feature_selection import RFE

estimator = lgb.LGBMClassifier(n_estimators=100, max_depth=3, random_state=42, verbose=-1)

rfe = RFE(estimator, n_features_to_select=10, step=1)
rfe.fit(X_new, y_new)

selected = X_new.columns[rfe.support_].tolist()
print("Отобрано 10 признаков:")
for f in selected:
    print(" -", f)

Отобрано 10 признаковы:
 - Pclass
 - Age
 - Fare
 - FamilySize
 - Fare_per_person
 - Pclass_x_FamilySize
 - Fare_relative
 - Sex_female
 - Embarked_S
 - Title_Master


In [19]:
X_selected = X_new[selected]

lgb_sel = lgb.LGBMClassifier(n_estimators=200, max_depth=3, learning_rate=0.1,
                              random_state=42, verbose=-1)
scores_sel = cross_val_score(lgb_sel, X_selected, y_new, cv=cv, scoring="accuracy")

print(f"BASELINE (15):     {scores_base.mean():.4f}")
print(f"С новыми (23):     {scores_new.mean():.4f}")
print(f"Отобрано 10:       {scores_sel.mean():.4f}")

BASELINE (15):     0.8440
С новыми (23):     0.8406
Отобрано 10:       0.8485


In [23]:
def eval_multi_seed(X, y, model_factory, seeds=[0, 42, 100, 7, 123]):
    accs = []
    for seed in seeds:
        cv_s = StratifiedKFold(n_splits=5, shuffle=True, random_state=seed)
        m = model_factory()
        accs.extend(cross_val_score(m, X, y, cv=cv_s, scoring="accuracy"))
    return np.mean(accs), np.std(accs)

lgb_factory = lambda: lgb.LGBMClassifier(n_estimators=200, max_depth=3,
                                          learning_rate=0.1, random_state=42,
                                          verbose=-1)

mean_base, std_base = eval_multi_seed(X_base, y, lgb_factory)
mean_new,  std_new  = eval_multi_seed(X_new,  y_new, lgb_factory)
mean_sel,  std_sel  = eval_multi_seed(X_selected, y_new, lgb_factory)

print(f"{'Набор':<20} {'Mean':<10} {'Std':<10}")
print("-" * 45)
print(f"{'Baseline (15)':<20} {mean_base:.4f}    {std_base:.4f}")
print(f"{'С новыми (22)':<20} {mean_new:.4f}    {std_new:.4f}")
print(f"{'Отобранные (10)':<20} {mean_sel:.4f}    {std_sel:.4f}")

Набор                Mean       Std       
---------------------------------------------
Baseline (15)        0.8402    0.0219
С новыми (22)        0.8384    0.0278
Отобранные (10)      0.8408    0.0264


In [ ]:
## Multi-seed сравнение (5 seeds × 5 folds)

| Набор | Mean | Std |
|---|---|---|
| Baseline (15) | 0.8402 | 0.0219 |
| С новыми (22) | 0.8384 | 0.0278 |
| Отобранные (10) | 0.8408 | 0.0264 |

### Выводы
1. Все три набора эквивалентны — разница < std.
2. Новые фичи не помогли, а добавили шум (std ↑ с 0.022 до 0.028).
3. Отобранные (10) — лучший компромисс: качество то же, фич в 2 раза меньше.
4. Берём 10 отобранных: проще, быстрее, не хуже.